# STRAT-003: Short-Term Strategy Exploration

**Goal:** Find a strategy that trades 10-20+ times per year for active paper trading.

**STRAT-002 Problem:** Only 8 trades in 7 years = can't paper test effectively.

**Ideas to explore:**
1. STH-SOPR extremes (more volatile than SOPR)
2. Shorter lookback z-scores (7-day, 14-day)
3. Mean reversion signals
4. Multiple weaker signals combined
5. Smaller trailing stops (faster exits)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import vectorbt as vbt
import warnings
warnings.filterwarnings('ignore')

print("STRAT-003: Short-Term Strategy Exploration 🔬")

In [ ]:
# Load all available data
DATA_DIR = Path("../data/daily")

price = pd.read_parquet(DATA_DIR / "price.parquet").rename(columns={"value": "price"}).set_index("time")
sopr = pd.read_parquet(DATA_DIR / "sopr.parquet").rename(columns={"value": "sopr"}).set_index("time")
sopr_sth = pd.read_parquet(DATA_DIR / "sopr_sth.parquet").rename(columns={"value": "sopr_sth"}).set_index("time")
mvrv = pd.read_parquet(DATA_DIR / "mvrv.parquet").rename(columns={"value": "mvrv"}).set_index("time")
realized_loss = pd.read_parquet(DATA_DIR / "realized_loss.parquet").rename(columns={"value": "realized_loss"}).set_index("time")

df = price.join(sopr, how='inner').join(sopr_sth, how='inner').join(mvrv, how='inner').join(realized_loss, how='inner')
df = df.sort_index()
df = df[df.index >= '2019-01-01'].dropna()

print(f"Data: {len(df)} rows ({df.index.min().date()} to {df.index.max().date()})")

---
## 1. Analyze Signal Frequency

In [ ]:
# Check how often different conditions occur
print("SIGNAL FREQUENCY ANALYSIS")
print("="*70)

# STRAT-002 signals (for reference)
df['rl_z30'] = (df['realized_loss'] - df['realized_loss'].rolling(30).mean()) / df['realized_loss'].rolling(30).std()
strat002 = (df['sopr'] < 1) & (df['sopr_sth'] < 1) & (df['rl_z30'] > 0.5)
strat002_first = strat002 & ~strat002.shift(1).fillna(False)

print(f"\nSTRAT-002 (current long-term):")
print(f"  Days condition met: {strat002.sum()} ({strat002.mean()*100:.1f}%)")
print(f"  Entry signals (first day): {strat002_first.sum()}")
print(f"  Signals per year: {strat002_first.sum() / 7:.1f}")

In [ ]:
# Test more frequent signals
print("\nALTERNATIVE SIGNAL FREQUENCIES:")
print("-"*70)

signals_to_test = [
    # Single conditions
    ("SOPR < 1", df['sopr'] < 1),
    ("SOPR < 0.98", df['sopr'] < 0.98),
    ("SOPR < 0.95", df['sopr'] < 0.95),
    ("STH-SOPR < 1", df['sopr_sth'] < 1),
    ("STH-SOPR < 0.98", df['sopr_sth'] < 0.98),
    ("STH-SOPR < 0.95", df['sopr_sth'] < 0.95),
    
    # Combined (less strict)
    ("SOPR < 1 OR STH-SOPR < 1", (df['sopr'] < 1) | (df['sopr_sth'] < 1)),
    ("SOPR < 1 AND STH-SOPR < 1", (df['sopr'] < 1) & (df['sopr_sth'] < 1)),
]

print(f"{'Signal':<35} {'Days':>10} {'%':>8} {'Entries':>10} {'Per Year':>10}")
print("-"*70)

for name, cond in signals_to_test:
    first_day = cond & ~cond.shift(1).fillna(False)
    print(f"{name:<35} {cond.sum():>10} {cond.mean()*100:>7.1f}% {first_day.sum():>10} {first_day.sum()/7:>10.1f}")

---
## 2. Short-Term Z-Scores

In [ ]:
# Create shorter lookback z-scores
for window in [7, 14, 21]:
    df[f'sopr_z{window}'] = (df['sopr'] - df['sopr'].rolling(window).mean()) / df['sopr'].rolling(window).std()
    df[f'sth_sopr_z{window}'] = (df['sopr_sth'] - df['sopr_sth'].rolling(window).mean()) / df['sopr_sth'].rolling(window).std()

print("SHORT-TERM Z-SCORE SIGNALS")
print("="*70)

zscore_signals = [
    # SOPR z-scores
    ("SOPR Z7 < -1", df['sopr_z7'] < -1),
    ("SOPR Z7 < -1.5", df['sopr_z7'] < -1.5),
    ("SOPR Z7 < -2", df['sopr_z7'] < -2),
    ("SOPR Z14 < -1", df['sopr_z14'] < -1),
    ("SOPR Z14 < -1.5", df['sopr_z14'] < -1.5),
    ("SOPR Z14 < -2", df['sopr_z14'] < -2),
    
    # STH-SOPR z-scores  
    ("STH-SOPR Z7 < -1", df['sth_sopr_z7'] < -1),
    ("STH-SOPR Z7 < -1.5", df['sth_sopr_z7'] < -1.5),
    ("STH-SOPR Z7 < -2", df['sth_sopr_z7'] < -2),
    ("STH-SOPR Z14 < -1", df['sth_sopr_z14'] < -1),
    ("STH-SOPR Z14 < -1.5", df['sth_sopr_z14'] < -1.5),
]

print(f"{'Signal':<25} {'Days':>10} {'%':>8} {'Entries':>10} {'Per Year':>10}")
print("-"*70)

for name, cond in zscore_signals:
    cond = cond.fillna(False)
    first_day = cond & ~cond.shift(1).fillna(False)
    print(f"{name:<25} {cond.sum():>10} {cond.mean()*100:>7.1f}% {first_day.sum():>10} {first_day.sum()/7:>10.1f}")

---
## 3. Quick Backtest Function

In [ ]:
def quick_backtest(data, entries, trail_pct=0.15, init_cash=100000):
    """Quick VectorBT backtest with shorter trail"""
    if entries.sum() == 0:
        return None
    
    pf = vbt.Portfolio.from_signals(
        close=data['price'],
        entries=entries,
        exits=None,
        sl_stop=trail_pct,
        sl_trail=True,
        stop_exit_price='close',
        fees=0.001,
        init_cash=init_cash,
        freq='D'
    )
    return pf

def get_metrics(pf, data):
    """Extract key metrics"""
    if pf is None or pf.trades.count() == 0:
        return None
    
    bh = (data['price'].iloc[-1] / data['price'].iloc[0]) - 1
    
    return {
        'return': pf.total_return(),
        'bh': bh,
        'beat': pf.total_return() > bh,
        'trades': pf.trades.count(),
        'win_rate': pf.trades.win_rate(),
        'sharpe': pf.sharpe_ratio(),
        'max_dd': pf.max_drawdown(),
        'trades_per_year': pf.trades.count() / 7
    }

---
## 4. Test Short-Term Strategies

In [ ]:
print("SHORT-TERM STRATEGY BACKTEST")
print("="*130)
print(f"{'Strategy':<40} {'Return':>10} {'B&H':>10} {'Beat':>6} {'Trades':>8} {'/Year':>6} {'Win%':>6} {'Sharpe':>8} {'MaxDD':>8}")
print("-"*130)

strategies = [
    # Reference: STRAT-002 with 30% trail
    ("STRAT-002 (30% trail)", strat002_first, 0.30),
    
    # Simple signals with shorter trails
    ("SOPR < 0.98 (15% trail)", (df['sopr'] < 0.98) & ~(df['sopr'] < 0.98).shift(1).fillna(False), 0.15),
    ("SOPR < 0.98 (10% trail)", (df['sopr'] < 0.98) & ~(df['sopr'] < 0.98).shift(1).fillna(False), 0.10),
    ("STH-SOPR < 0.98 (15% trail)", (df['sopr_sth'] < 0.98) & ~(df['sopr_sth'] < 0.98).shift(1).fillna(False), 0.15),
    ("STH-SOPR < 0.98 (10% trail)", (df['sopr_sth'] < 0.98) & ~(df['sopr_sth'] < 0.98).shift(1).fillna(False), 0.10),
    
    # Z-score signals
    ("SOPR Z7 < -1.5 (15% trail)", (df['sopr_z7'] < -1.5) & ~(df['sopr_z7'] < -1.5).shift(1).fillna(False), 0.15),
    ("SOPR Z14 < -1.5 (15% trail)", (df['sopr_z14'] < -1.5) & ~(df['sopr_z14'] < -1.5).shift(1).fillna(False), 0.15),
    ("STH-SOPR Z7 < -1.5 (15% trail)", (df['sth_sopr_z7'] < -1.5) & ~(df['sth_sopr_z7'] < -1.5).shift(1).fillna(False), 0.15),
    ("STH-SOPR Z14 < -1.5 (15% trail)", (df['sth_sopr_z14'] < -1.5) & ~(df['sth_sopr_z14'] < -1.5).shift(1).fillna(False), 0.15),
    
    # Combined short-term
    ("SOPR<1 & STH<1 (15% trail)", ((df['sopr'] < 1) & (df['sopr_sth'] < 1)) & ~((df['sopr'] < 1) & (df['sopr_sth'] < 1)).shift(1).fillna(False), 0.15),
    ("SOPR<1 & STH<1 (10% trail)", ((df['sopr'] < 1) & (df['sopr_sth'] < 1)) & ~((df['sopr'] < 1) & (df['sopr_sth'] < 1)).shift(1).fillna(False), 0.10),
]

results = []
for name, entries, trail in strategies:
    entries = entries.fillna(False)
    pf = quick_backtest(df, entries, trail)
    m = get_metrics(pf, df)
    
    if m:
        beat = '✅' if m['beat'] else '❌'
        print(f"{name:<40} {m['return']*100:>+9.0f}% {m['bh']*100:>+9.0f}% {beat:>6} {m['trades']:>8} {m['trades_per_year']:>6.1f} {m['win_rate']*100:>5.0f}% {m['sharpe']:>8.2f} {m['max_dd']*100:>7.0f}%")
        results.append({'name': name, 'metrics': m, 'trail': trail})
    else:
        print(f"{name:<40} {'No trades':>10}")

In [ ]:
# Find best short-term strategies (>10 trades/year)
print("\n" + "="*70)
print("BEST SHORT-TERM STRATEGIES (>10 trades/year)")
print("="*70)

short_term = [r for r in results if r['metrics']['trades_per_year'] >= 10]
short_term = sorted(short_term, key=lambda x: x['metrics']['return'], reverse=True)

for i, r in enumerate(short_term[:5]):
    m = r['metrics']
    print(f"\n{i+1}. {r['name']}")
    print(f"   Return: {m['return']*100:+,.0f}% | Trades: {m['trades']} ({m['trades_per_year']:.1f}/yr)")
    print(f"   Win Rate: {m['win_rate']*100:.0f}% | Sharpe: {m['sharpe']:.2f} | MaxDD: {m['max_dd']*100:.0f}%")
    print(f"   Beat B&H: {'✅' if m['beat'] else '❌'}")

---
## 5. Deep Dive Best Short-Term Strategy

In [ ]:
# Pick the best short-term strategy and analyze
if short_term:
    best = short_term[0]
    print(f"\nDEEP DIVE: {best['name']}")
    print("="*80)
    
    # Recreate the backtest
    if 'SOPR<1 & STH<1' in best['name']:
        cond = (df['sopr'] < 1) & (df['sopr_sth'] < 1)
    elif 'STH-SOPR < 0.98' in best['name']:
        cond = df['sopr_sth'] < 0.98
    elif 'SOPR < 0.98' in best['name']:
        cond = df['sopr'] < 0.98
    elif 'STH-SOPR Z' in best['name']:
        cond = df['sth_sopr_z14'] < -1.5
    else:
        cond = df['sopr_z14'] < -1.5
    
    entries = cond & ~cond.shift(1).fillna(False)
    pf = quick_backtest(df, entries, best['trail'])
    
    print(pf.stats())

In [ ]:
# Trade log
if short_term and pf:
    print("\nTRADE LOG")
    print("="*80)
    print(pf.trades.records_readable.to_string())

In [ ]:
# Year by year
if short_term and pf:
    print("\nYEAR-BY-YEAR PERFORMANCE")
    print("="*80)
    
    equity = pf.value()
    
    for year in [2019, 2020, 2021, 2022, 2023, 2024, 2025]:
        year_eq = equity[(equity.index >= f'{year}-01-01') & (equity.index <= f'{year}-12-31')]
        year_price = df[(df.index >= f'{year}-01-01') & (df.index <= f'{year}-12-31')]['price']
        
        if len(year_eq) > 0 and len(year_price) > 0:
            strat = (year_eq.iloc[-1] / year_eq.iloc[0] - 1) * 100
            bh = (year_price.iloc[-1] / year_price.iloc[0] - 1) * 100
            beat = '✅' if strat > bh else '❌'
            print(f"{year}: Strategy {strat:+6.0f}% | B&H {bh:+6.0f}% | {beat}")

In [ ]:
# Plot
if short_term and pf:
    pf.plot().show()

---
## 6. Summary

In [ ]:
print("\n" + "="*70)
print("SUMMARY")
print("="*70)

print("\n📊 STRAT-002 (Long-Term):")
print("   Entry: SOPR<1 & STH-SOPR<1 & RL Z>0.5")
print("   Exit: 30% trailing stop")
print("   Frequency: ~1-2 trades/year")
print("   Result: +5,754% (validated)")
print("   Use: Deploy and forget")

if short_term:
    best = short_term[0]
    m = best['metrics']
    print(f"\n📈 STRAT-003 (Short-Term) - BEST CANDIDATE:")
    print(f"   Strategy: {best['name']}")
    print(f"   Exit: {best['trail']*100:.0f}% trailing stop")
    print(f"   Frequency: {m['trades_per_year']:.0f} trades/year")
    print(f"   Result: {m['return']*100:+,.0f}%")
    print(f"   Win Rate: {m['win_rate']*100:.0f}%")
    print(f"   Use: Paper trading")